# 01 - Exploratory analysis

What the lifecycle data says about pricing, conversion and overrides, before any
model is fitted. Run `python scripts/train_all.py` first if the dataset is missing.

The dataset is synthetic (`src/aavas/synthetic/generator.py`). It carries latent
`true_*` columns that real data would never expose, which is what lets the
notebooks below check whether the models recover the structure that was actually there.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

from aavas.ingestion import loader
df = loader.load_training_table()
print(f"{len(df):,} applications, {df.application_date.min():%b %Y} to {df.application_date.max():%b %Y}")


## The pricing funnel

Every difference between these numbers is a learning signal.

In [ ]:
funnel = df[["policy_card_rate", "ai_recommended_rate", "sales_offered_rate", "sanctioned_rate"]].mean()
print(funnel.round(3).to_string())
print()
print(f"Sanction-to-disbursement: {df.disbursed.mean():.1%}")
print(f"Overridden by a human:    {df.was_overridden.mean():.1%}")
print(f"Balance transfer in 24m:  {df.loc[df.disbursed, 'bt_out_24m'].mean():.1%}")


## Price sensitivity is not uniform

The same 25 bps does very different things to different customers. This is the
heterogeneity the conversion model has to recover from observable features.

In [ ]:
df["gap_bps"] = (df.sanctioned_rate - df.competitor_rate) * 100
by_gap = df.groupby(pd.cut(df.gap_bps, [-200, -25, 0, 25, 50, 75, 200]), observed=True).agg(
    n=("loan_id", "size"), s2d=("disbursed", "mean"), rate=("sanctioned_rate", "mean"))
print(by_gap.round(3).to_string())
print()
print("Latent elasticity by segment (points of conversion lost per 100 bps):")
print(df.groupby("segment").price_elasticity.agg(["mean", "std"]).round(2).to_string())


## Why price uplift is not yield

A higher rate raises the contractual spread and shortens the loan's life. The
lifetime value column nets the two off.

In [ ]:
disbursed = df[df.disbursed]
print(disbursed.groupby("bt_out_24m")[["sanctioned_rate", "realized_life_years",
                                       "realized_yield_pct", "realized_value_pct"]].mean().round(2).to_string())
print()
print("Lifetime value by rate decile (% of disbursal):")
print(disbursed.groupby(pd.qcut(disbursed.sanctioned_rate, 5), observed=True)
      .agg(rate=("sanctioned_rate", "mean"), value=("realized_value_pct", "mean"),
           bt=("bt_out_24m", "mean")).round(2).to_string())


## Overrides: the confounding problem

The naive comparison says discounts hurt conversion. They do not - salespeople
discount the files that were hardest to convert in the first place. Notebook
`03` and `src/aavas/learning/override_learning.py` do this properly.

In [ ]:
print(df.groupby("override_direction").agg(
    n=("loan_id", "size"), s2d=("disbursed", "mean"),
    value=("realized_value_pct", "mean"), rate=("sanctioned_rate", "mean")).round(3).to_string())
print()
print("Override reasons:")
print(df[df.was_overridden].override_reason.value_counts(normalize=True).round(3).to_string())


## Regional structure

Competitive intensity differs by state. This is what the phase-2 federated
module's regional personalization exists to capture.

In [ ]:
print(df.groupby("state").agg(
    n=("loan_id", "size"), card=("policy_card_rate", "mean"), sanctioned=("sanctioned_rate", "mean"),
    discount_bps=("total_discount_bps", "mean"), s2d=("disbursed", "mean")).round(2).to_string())
